In [1]:
import warnings
warnings.filterwarnings("ignore")
import sys
import os
import torch
sys.path.append(rf'/home/{os.getlogin()}/data_share')
import FactorFramework.FactorFramework as ff

# 添加基础字段的步骤

In [2]:
# 1. 参考get_base_data_fea.py计算fea格式的基础字段数据。计算完成后，data_share/Stock60sBaseDataAll/Feather下应该有以改字段系列名称命名的子目录，子目录中包含【日期.fea】格式的每日基础字段文件

In [3]:
# 2. 使用ff.fea_to_mmap将fea格式的文件转换为内存映射文件
# 第一个参数是基础字段系列名，conv_start_date和conv_end_date是转换的起止日期
ff.fea_to_mmap_user('Test1', conv_start_date='20210104', conv_end_date='20221230')

update feather data of "Test1" to mmap
start 20210104, end 20221230, total 485 days


fill mmap: 100%|██████████| 485/485 [35:34<00:00,  4.40s/it]

[success] updating process is finished


In [4]:
# 3. 转换完成后，可以确认一下新的字段是否已经注册
ff.load_all_base_data() # 重新读取最新的基础字段文件
ff.base_data_config.Base_data_dict.keys() # 显示所有基础字段

dict_keys(['Volbar_price', 'Volbar_bn_5', 'Volbar_sv_9', 'Volbar_sv_7', 'Volbar_sp_8', 'Volbar_sn_8', 'Volbar_bn_3', 'Volbar_bn_4', 'Volbar_bp_9', 'Volbar_sp_6', 'Volbar_bn_2', 'Volbar_sv_6', 'Volbar_bp_2', 'Volbar_bv_6', 'Volbar_bp_5', 'Volbar_bv_8', 'Volbar_sp_7', 'Volbar_sn_4', 'Volbar_sn_2', 'Volbar_sv_3', 'Volbar_bv_4', 'Volbar_bn_1', 'Volbar_bp_4', 'Volbar_bp_3', 'Volbar_sp_5', 'Volbar_bp_7', 'Volbar_sv_2', 'Volbar_sv_8', 'Volbar_sv_4', 'Volbar_bv_3', 'Volbar_bv_10', 'Volbar_sn_7', 'Volbar_sp_4', 'Volbar_sp_2', 'Volbar_bv_5', 'Volbar_bn_8', 'Volbar_bv_7', 'Volbar_bp_8', 'Volbar_sv_1', 'Volbar_bp_6', 'Volbar_bp_1', 'Volbar_bn_6', 'Volbar_sp_10', 'Volbar_sn_5', 'Volbar_sn_9', 'Volbar_sn_3', 'Volbar_bn_9', 'Volbar_sn_10', 'Volbar_bv_9', 'Volbar_sn_1', 'Volbar_sp_3', 'Volbar_sp_9', 'Volbar_bv_2', 'Volbar_sv_5', 'Volbar_sp_1', 'Volbar_bv_1', 'Volbar_sv_10', 'Volbar_bp_10', 'Volbar_sn_6', 'Volbar_bn_10', 'Volbar_bn_7', '50ms_TradePriceStd', '50ms_TradePriceMean_B', '50ms_TradeAmtSum_B'

# 添加算子的步骤

In [5]:
# 1. 定义算子函数
# 框架中涉及到基础字段和其他算子的返回值类型均为torch.Tensor
# 入参和返回的Tensor的0轴是时间，1轴是股票，返回值应当与入参Tensor形状相同
# 算子的入参根据算子的功能可能是Tensor，int，float，str等，但一定会有至少一个Tensor
# 时序算子请以ts_前缀命名，截面算子请以cs_前缀命名，其他逐元素算子（例如加减乘除）不需要时序或截面前缀

In [6]:
# 时序算子
# 以ts_corr_sample算子为例，这个算子计算x和y在过去d分钟的相关系数，是每个股票在自身时序上的相关系数
# 时序算子最开始的d-1分钟因为数据量不足，结果应当为空值，d分钟开始有非空值
# 不需要在时序算子的前面d-1分钟填补空值。实际计算时，框架会使用当天截止当前分钟的可得数据或昨日数据来避免空值问题，在算子中填补前d-1分钟的空值会造成干扰
def ts_corr_sample(x: torch.Tensor, y: torch.Tensor, d: int) -> torch.Tensor:
    res = torch.full(x.shape, torch.nan, dtype=x.dtype, device=x.device)
    x_unfold = x.unfold(0, d, 1)
    y_unfold = y.unfold(0, d, 1)
    x_demean = x_unfold - x_unfold.nanmean(dim=-1, keepdim=True)
    y_demean = y_unfold - y_unfold.nanmean(dim=-1, keepdim=True)
    x_std = torch.sqrt(torch.nansum(torch.pow(x_demean, 2), dim=-1))
    y_std = torch.sqrt(torch.nansum(torch.pow(y_demean, 2), dim=-1))
    numerator = (x_demean * y_demean).nansum(dim=-1)
    denominator = x_std * y_std
    res[d - 1:] = numerator / denominator
    res[d - 1:][(x_std < 1e-9) | (y_std < 1e-9)] = torch.nan
    return res

In [7]:
# 截面算子
# 以cs_rank_sample算子为例，这个算子计算所有股票在每分钟的数值x的排名
# 截面算子需要注意每个截面上的股票可比问题，往往需要统计valid_mask为True的股票，只有这些股票参与横截面计算
# valid_mask默认为全部x的值非空的股票（这是对x的值在横截面排序，仅对非空值排序是自然的要求）
def cs_rank_sample(x: torch.Tensor, valid_mask: any = None, pct: bool = True) -> torch.Tensor:
    ranks = x.clone()
    if valid_mask is None:
        valid_mask = torch.isfinite(x)
    ranks[~valid_mask] = torch.finfo(x.dtype).max
    ranks = torch.argsort(torch.argsort(ranks, dim=-1), dim=-1)
    ranks = ranks.to(dtype=x.dtype) + 1
    if pct:
        valid_counts = valid_mask.sum(dim=-1, keepdim=True)
        ranks = ranks / valid_counts
    ranks[~valid_mask] = torch.nan
    return ranks

In [8]:
# 2. 测试算子正确性
x = torch.randn([10, 5])
y = torch.randn([10, 5])
d = 3

In [9]:
ts_corr_sample(x, y, d)

tensor([[    nan,     nan,     nan,     nan,     nan],
        [    nan,     nan,     nan,     nan,     nan],
        [-0.9630,  0.5847,  0.8362,  0.6403, -0.9503],
        [-0.9378, -0.4985,  0.8930, -0.3109, -0.9499],
        [-0.8949,  0.0388,  0.8681, -0.9998, -0.9110],
        [-0.7669, -0.8368,  0.8359, -0.9112, -0.4822],
        [-0.5231, -0.9960,  0.8569, -0.5651, -0.1032],
        [ 0.0519,  0.2001, -0.5805,  0.0518,  0.9480],
        [ 0.0774, -0.2035,  0.3220,  0.6130, -0.7413],
        [ 0.6690,  0.8700,  0.3856,  0.9561, -0.9595]])

In [10]:
cs_rank_sample(x, None, True)

tensor([[0.6000, 1.0000, 0.4000, 0.8000, 0.2000],
        [0.8000, 0.6000, 1.0000, 0.2000, 0.4000],
        [0.6000, 1.0000, 0.4000, 0.8000, 0.2000],
        [0.4000, 0.2000, 1.0000, 0.8000, 0.6000],
        [1.0000, 0.4000, 0.8000, 0.6000, 0.2000],
        [0.2000, 0.6000, 0.8000, 0.4000, 1.0000],
        [0.4000, 0.2000, 1.0000, 0.8000, 0.6000],
        [0.4000, 0.8000, 0.6000, 0.2000, 1.0000],
        [0.2000, 0.8000, 0.6000, 1.0000, 0.4000],
        [0.4000, 1.0000, 0.6000, 0.2000, 0.8000]])

In [11]:
# 3. 确认算子逻辑无误后，按算子类别合并代码至FactorFramework下的torch_operators_all.py